In [0]:
# ============================================================
# GOLD ANALYTICS — INITIALIZATION
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("""
CREATE SCHEMA IF NOT EXISTS telecom.gold
""")

print("✅ Gold schema ready: telecom.gold")
print("🚀 Starting Gold analytical layer")

In [0]:
# ============================================================
# GOLD — DIM DATE
# ============================================================

from datetime import date, timedelta

start_date = date(2025, 1, 1)
end_date = date(2027, 12, 31)

date_rows = []
current_date = start_date

while current_date <= end_date:
    date_rows.append({
        "date_key": int(current_date.strftime("%Y%m%d")),
        "full_date": current_date,
        "day": current_date.day,
        "month": current_date.month,
        "month_name": current_date.strftime("%B"),
        "quarter": ((current_date.month - 1) // 3) + 1,
        "quarter_name": f"Q{((current_date.month - 1) // 3) + 1}",
        "year": current_date.year,
        "day_of_week": current_date.isoweekday(),
        "day_name": current_date.strftime("%A"),
        "week_of_year": current_date.isocalendar()[1],
        "is_weekend": current_date.isoweekday() >= 6
    })

    current_date += timedelta(days=1)

dim_date = spark.createDataFrame(date_rows)

(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_date")
)

print(
    f"✅ dim_date created: {dim_date.count():,} records"
)

display(dim_date.limit(10))

In [0]:
# ============================================================
# GOLD — DIM CUSTOMER
# ============================================================

customers_silver = spark.table("telecom.silver.customers")

dim_customer = (
    customers_silver
    .withColumn(
        "customer_name",
        F.concat_ws(
            " ",
            F.col("first_name"),
            F.col("last_name")
        )
    )
    .withColumn(
        "age",
        F.floor(
            F.datediff(
                F.current_date(),
                F.col("date_of_birth")
            ) / 365.25
        ).cast("int")
    )
    .withColumn(
        "customer_tenure_days",
        F.datediff(
            F.current_date(),
            F.to_date("registration_date")
        )
    )
    .withColumn(
        "customer_tenure_years",
        F.round(
            F.col("customer_tenure_days") / 365.25,
            2
        )
    )
    .select(
        F.monotonically_increasing_id().alias("customer_key"),
        "customer_id",
        "customer_name",
        "first_name",
        "last_name",
        "date_of_birth",
        "age",
        "gender",
        "email",
        "registration_date",
        "customer_status",
        "customer_tenure_days",
        "customer_tenure_years",
        "created_at",
        "updated_at"
    )
)

(
    dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_customer")
)

print(
    f"✅ dim_customer created: "
    f"{dim_customer.count():,} records"
)

display(dim_customer.limit(10))

In [0]:
# ============================================================
# GOLD — DIM PLAN
# ============================================================

plans_silver = spark.table("telecom.silver.mobile_plans")

dim_plan = (
    plans_silver
    .select(
        F.monotonically_increasing_id().alias("plan_key"),
        "plan_id",
        "plan_name",
        "plan_type",
        "monthly_charge",
        "plan_status",
        "created_at",
        "updated_at"
    )
)

(
    dim_plan.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_plan")
)

print(
    f"✅ dim_plan created: "
    f"{dim_plan.count():,} records"
)

display(dim_plan)

In [0]:
# ============================================================
# GOLD — DIM SERVICE
# ============================================================

services_silver = spark.table("telecom.silver.service_types")

dim_service = (
    services_silver
    .select(
        F.monotonically_increasing_id().alias("service_key"),
        "service_id",
        "service_name",
        "service_category",
        "description",
        "created_at"
    )
)

(
    dim_service.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_service")
)

print(
    f"✅ dim_service created: "
    f"{dim_service.count():,} records"
)

display(dim_service)

In [0]:
# ============================================================
# GOLD — DIM SUBSCRIPTION
# ============================================================

subscriptions_silver = spark.table(
    "telecom.silver.subscriptions"
)

plans_silver = spark.table(
    "telecom.silver.mobile_plans"
)

dim_subscription = (
    subscriptions_silver.alias("s")
    .join(
        plans_silver.alias("p"),
        F.col("s.plan_id") == F.col("p.plan_id"),
        "left"
    )
    .select(
        F.monotonically_increasing_id().alias("subscription_key"),
        F.col("s.subscription_id"),
        F.col("s.customer_id"),
        F.col("s.plan_id"),
        F.col("p.plan_name"),
        F.col("p.plan_type"),
        F.col("p.monthly_charge"),
        F.col("s.phone_number"),
        F.col("s.subscription_status"),
        F.col("s.activation_date"),
        F.col("s.deactivation_date"),
        F.col("s.created_at"),
        F.col("s.updated_at")
    )
)

(
    dim_subscription.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_subscription")
)

print(
    f"✅ dim_subscription created: "
    f"{dim_subscription.count():,} records"
)

display(dim_subscription.limit(10))

In [0]:
# ============================================================
# GOLD — DIM SERVICE AREA
# ============================================================

service_areas_silver = spark.table(
    "telecom.silver.service_areas"
)

dim_service_area = (
    service_areas_silver
    .select(
        F.monotonically_increasing_id().alias("area_key"),
        "area_id",
        "area_name",
        "city",
        "state",
        "region",
        "network_coverage",
        "created_at"
    )
)

(
    dim_service_area.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.dim_service_area")
)

print(
    f"✅ dim_service_area created: "
    f"{dim_service_area.count():,} records"
)

display(dim_service_area.limit(10))

In [0]:
# ============================================================
# GOLD — FACT CALL USAGE
# ============================================================

calls_silver = spark.table(
    "telecom.silver.call_records"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_call_usage = (
    calls_silver.alias("c")
    .join(
        dim_subscription_gold.alias("s"),
        F.col("c.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )
    .join(
        dim_date_gold.alias("d"),
        F.to_date(F.col("c.call_start_time")) ==
        F.col("d.full_date"),
        "left"
    )
    .select(
        F.col("c.call_id").alias("call_id"),
        F.col("s.subscription_key").alias("subscription_key"),
        F.col("s.customer_id").alias("customer_id"),
        F.col("d.date_key").alias("date_key"),
        F.col("c.subscription_id").alias("subscription_id"),
        F.col("c.call_type").alias("call_type"),
        F.col("c.call_direction").alias("call_direction"),
        F.col("c.destination_number").alias("destination_number"),
        F.col("c.call_start_time").alias("call_start_time"),
        F.col("c.call_end_time").alias("call_end_time"),
        F.col("c.call_duration_seconds").alias("call_duration_seconds"),
        F.col("c.call_charges").alias("call_charges"),
        F.col("c.created_at").alias("created_at")
    )
)

(
    fact_call_usage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_call_usage")
)

print(
    f"✅ fact_call_usage created: "
    f"{fact_call_usage.count():,} records"
)

display(fact_call_usage.limit(10))

In [0]:
# ============================================================
# GOLD — FACT SMS USAGE
# ============================================================

sms_silver = spark.table(
    "telecom.silver.sms_records"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_sms_usage = (
    sms_silver.alias("sms")
    .join(
        dim_subscription_gold.alias("s"),
        F.col("sms.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )
    .join(
        dim_date_gold.alias("d"),
        F.to_date(F.col("sms.sms_timestamp")) ==
        F.col("d.full_date"),
        "left"
    )
    .select(
        F.col("sms.sms_id").alias("sms_id"),
        F.col("s.subscription_key").alias("subscription_key"),
        F.col("s.customer_id").alias("customer_id"),
        F.col("d.date_key").alias("date_key"),
        F.col("sms.subscription_id").alias("subscription_id"),
        F.col("sms.sms_type").alias("sms_type"),
        F.col("sms.sms_direction").alias("sms_direction"),
        F.col("sms.destination_number").alias("destination_number"),
        F.col("sms.sms_timestamp").alias("sms_timestamp"),
        F.col("sms.sms_count").alias("sms_count"),
        F.col("sms.sms_charges").alias("sms_charges"),
        F.col("sms.created_at").alias("created_at")
    )
)

(
    fact_sms_usage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_sms_usage")
)

print(
    f"✅ fact_sms_usage created: "
    f"{fact_sms_usage.count():,} records"
)

display(fact_sms_usage.limit(10))

In [0]:
# ============================================================
# GOLD — FACT DATA USAGE
# ============================================================

data_usage_silver = spark.table(
    "telecom.silver.data_usage"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_data_usage = (
    data_usage_silver.alias("u")
    .join(
        dim_subscription_gold.alias("s"),
        F.col("u.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )
    .join(
        dim_date_gold.alias("d"),
        F.col("u.usage_date") ==
        F.col("d.full_date"),
        "left"
    )
    .select(
        F.col("u.usage_id").alias("usage_id"),
        F.col("s.subscription_key").alias("subscription_key"),
        F.col("s.customer_id").alias("customer_id"),
        F.col("d.date_key").alias("date_key"),
        F.col("u.subscription_id").alias("subscription_id"),
        F.col("u.usage_date").alias("usage_date"),
        F.col("u.usage_start_time").alias("usage_start_time"),
        F.col("u.usage_end_time").alias("usage_end_time"),
        F.col("u.data_consumed_mb").alias("data_consumed_mb"),
        F.col("u.data_charges").alias("data_charges"),
        F.col("u.network_type").alias("network_type"),
        F.col("u.created_at").alias("created_at")
    )
)

(
    fact_data_usage.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_data_usage")
)

print(
    f"✅ fact_data_usage created: "
    f"{fact_data_usage.count():,} records"
)

display(fact_data_usage.limit(10))

In [0]:
# ============================================================
# GOLD — FACT BILLING
# ============================================================

bills_silver = spark.table(
    "telecom.silver.bills"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_billing = (
    bills_silver.alias("b")
    .join(
        dim_subscription_gold.alias("s"),
        F.col("b.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )
    .join(
        dim_date_gold.alias("d"),
        F.col("b.bill_date") ==
        F.col("d.full_date"),
        "left"
    )
    .select(
        F.col("b.bill_id").alias("bill_id"),
        F.col("s.subscription_key").alias("subscription_key"),
        F.col("s.customer_id").alias("customer_id"),
        F.col("s.plan_id").alias("plan_id"),
        F.col("d.date_key").alias("date_key"),

        F.col("b.subscription_id").alias("subscription_id"),
        F.col("b.bill_date").alias("bill_date"),
        F.col("b.billing_period_start").alias("billing_period_start"),
        F.col("b.billing_period_end").alias("billing_period_end"),

        F.col("b.total_amount").alias("total_amount"),
        F.col("b.tax_amount").alias("tax_amount"),
        F.col("b.discount_amount").alias("discount_amount"),
        F.col("b.net_amount").alias("net_amount"),

        F.col("b.due_date").alias("due_date"),
        F.col("b.bill_status").alias("bill_status"),
        F.col("b.created_at").alias("created_at")
    )
)

(
    fact_billing.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_billing")
)

print(
    f"✅ fact_billing created: "
    f"{fact_billing.count():,} records"
)

display(fact_billing.limit(10))

In [0]:
# ============================================================
# GOLD — FACT PAYMENTS — CORRECTED
# ============================================================

payments_silver = spark.table(
    "telecom.silver.payments"
)

bills_silver = spark.table(
    "telecom.silver.bills"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_payments = (
    payments_silver.alias("p")

    # Payment → Bill
    .join(
        bills_silver.alias("b"),
        F.col("p.bill_id") == F.col("b.bill_id"),
        "left"
    )

    # Bill → Subscription
    .join(
        dim_subscription_gold.alias("s"),
        F.col("b.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )

    # Payment → Date
    .join(
        dim_date_gold.alias("d"),
        F.to_date(F.col("p.payment_date")) ==
        F.col("d.full_date"),
        "left"
    )

    .select(
        F.col("p.payment_id").alias("payment_id"),
        F.col("p.bill_id").alias("bill_id"),

        F.col("s.subscription_key").alias(
            "subscription_key"
        ),

        F.col("p.customer_id").alias("customer_id"),

        F.col("d.date_key").alias("date_key"),

        F.col("p.payment_date").alias(
            "payment_date"
        ),

        F.col("p.payment_amount").alias(
            "payment_amount"
        ),

        F.col("p.payment_method").alias(
            "payment_method"
        ),

        F.col("p.payment_status").alias(
            "payment_status"
        ),

        F.col("p.transaction_reference").alias(
            "transaction_reference"
        ),

        F.col("p.created_at").alias(
            "created_at"
        )
    )
)

# ------------------------------------------------------------
# SAFETY CHECK BEFORE WRITING
# ------------------------------------------------------------

source_count = payments_silver.count()
fact_count = fact_payments.count()

print(f"Source payment records : {source_count:,}")
print(f"Fact payment records   : {fact_count:,}")

if fact_count != source_count:
    raise Exception(
        f"❌ Row-count mismatch! "
        f"Source={source_count}, Fact={fact_count}"
    )

# ------------------------------------------------------------
# WRITE GOLD TABLE
# ------------------------------------------------------------

(
    fact_payments.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_payments")
)

print(
    f"✅ fact_payments created correctly: "
    f"{fact_count:,} records"
)

display(fact_payments.limit(10))

In [0]:
# ============================================================
# GOLD — FACT COMPLAINTS
# ============================================================

complaints_silver = spark.table(
    "telecom.silver.complaints"
)

dim_subscription_gold = spark.table(
    "telecom.gold.dim_subscription"
)

dim_date_gold = spark.table(
    "telecom.gold.dim_date"
)

fact_complaints = (
    complaints_silver.alias("c")

    # Complaint → Subscription
    .join(
        dim_subscription_gold.alias("s"),
        F.col("c.subscription_id") ==
        F.col("s.subscription_id"),
        "left"
    )

    # Complaint → Date
    .join(
        dim_date_gold.alias("d"),
        F.to_date(F.col("c.complaint_date")) ==
        F.col("d.full_date"),
        "left"
    )

    .select(
        F.col("c.complaint_id").alias("complaint_id"),

        F.col("s.subscription_key").alias(
            "subscription_key"
        ),

        F.col("c.customer_id").alias("customer_id"),

        F.col("d.date_key").alias("date_key"),

        F.col("c.subscription_id").alias(
            "subscription_id"
        ),

        F.col("c.category_id").alias(
            "category_id"
        ),

        F.col("c.complaint_date").alias(
            "complaint_date"
        ),

        F.col("c.complaint_description").alias(
            "complaint_description"
        ),

        F.col("c.complaint_status").alias(
            "complaint_status"
        ),

        F.col("c.priority").alias(
            "priority"
        ),

        F.col("c.resolution_date").alias(
            "resolution_date"
        ),

        F.col("c.resolution_notes").alias(
            "resolution_notes"
        ),

        F.col("c.created_at").alias(
            "created_at"
        ),

        F.col("c.updated_at").alias(
            "updated_at"
        )
    )
)

# ------------------------------------------------------------
# GRAIN SAFETY CHECK
# ------------------------------------------------------------

source_count = complaints_silver.count()
fact_count = fact_complaints.count()

print(f"Source complaint records : {source_count:,}")
print(f"Fact complaint records   : {fact_count:,}")

if fact_count != source_count:
    raise Exception(
        f"❌ Row-count mismatch! "
        f"Source={source_count}, Fact={fact_count}"
    )

# ------------------------------------------------------------
# WRITE GOLD TABLE
# ------------------------------------------------------------

(
    fact_complaints.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.fact_complaints")
)

print(
    f"✅ fact_complaints created correctly: "
    f"{fact_count:,} records"
)

display(fact_complaints.limit(10))

In [0]:
# ============================================================
# GOLD LAYER — FACT & DIMENSION VALIDATION
# ============================================================

expected_gold_counts = {
    "dim_date": 1095,
    "dim_customer": 1000,
    "dim_plan": 5,
    "dim_service": 10,
    "dim_subscription": 1242,
    "dim_service_area": 50,

    "fact_call_usage": 50000,
    "fact_sms_usage": 100000,
    "fact_data_usage": 75000,
    "fact_billing": 5039,
    "fact_payments": 4074,
    "fact_complaints": 500
}

gold_validation = []

for table_name, expected_count in expected_gold_counts.items():

    full_table_name = f"telecom.gold.{table_name}"

    try:
        actual_count = spark.table(full_table_name).count()

        gold_validation.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": actual_count,
            "status": (
                "PASS"
                if actual_count == expected_count
                else "CHECK"
            )
        })

    except Exception:

        gold_validation.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": -1,
            "status": "MISSING"
        })

gold_validation_df = spark.createDataFrame(gold_validation)

display(
    gold_validation_df.orderBy("table_name")
)

print("=" * 70)
print("GOLD LAYER VALIDATION")
print("=" * 70)

total_tables = len(gold_validation)

passed_tables = sum(
    1
    for row in gold_validation
    if row["status"] == "PASS"
)

print(f"Expected tables : {total_tables}")
print(f"Passed tables   : {passed_tables}")
print("=" * 70)

In [0]:
# ============================================================
# GOLD — CUSTOMER 360
# ============================================================

# ------------------------------------------------------------
# 1. CUSTOMER BASE
# ------------------------------------------------------------

customer_base = (
    spark.table("telecom.gold.dim_customer")
    .select(
        "customer_key",
        "customer_id",
        "customer_name",
        "age",
        "gender",
        "customer_status",
        "registration_date",
        "customer_tenure_days",
        "customer_tenure_years"
    )
)


# ------------------------------------------------------------
# 2. SUBSCRIPTION METRICS
# ------------------------------------------------------------

subscription_metrics = (
    spark.table("telecom.gold.dim_subscription")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("subscription_id").alias(
            "total_subscriptions"
        ),

        F.sum(
            F.when(
                F.col("subscription_status") == "ACTIVE",
                1
            ).otherwise(0)
        ).alias(
            "active_subscriptions"
        ),

        F.sum(
            F.when(
                F.col("subscription_status") == "CANCELLED",
                1
            ).otherwise(0)
        ).alias(
            "cancelled_subscriptions"
        ),

        F.first("plan_name", ignorenulls=True).alias(
            "current_plan"
        ),

        F.first("plan_type", ignorenulls=True).alias(
            "current_plan_type"
        )
    )
)


# ------------------------------------------------------------
# 3. CALL USAGE METRICS
# ------------------------------------------------------------

call_metrics = (
    spark.table("telecom.gold.fact_call_usage")
    .groupBy("customer_id")
    .agg(
        F.count("*").alias(
            "total_calls"
        ),

        F.round(
            F.sum("call_duration_seconds") / 60,
            2
        ).alias(
            "total_call_minutes"
        ),

        F.round(
            F.sum("call_charges"),
            2
        ).alias(
            "total_call_charges"
        )
    )
)


# ------------------------------------------------------------
# 4. SMS USAGE METRICS
# ------------------------------------------------------------

sms_metrics = (
    spark.table("telecom.gold.fact_sms_usage")
    .groupBy("customer_id")
    .agg(
        F.sum("sms_count").alias(
            "total_sms"
        ),

        F.round(
            F.sum("sms_charges"),
            2
        ).alias(
            "total_sms_charges"
        )
    )
)


# ------------------------------------------------------------
# 5. DATA USAGE METRICS
# ------------------------------------------------------------

data_metrics = (
    spark.table("telecom.gold.fact_data_usage")
    .groupBy("customer_id")
    .agg(
        F.round(
            F.sum("data_consumed_mb"),
            2
        ).alias(
            "total_data_mb"
        ),

        F.round(
            F.sum("data_consumed_mb") / 1024,
            2
        ).alias(
            "total_data_gb"
        ),

        F.round(
            F.sum("data_charges"),
            2
        ).alias(
            "total_data_charges"
        )
    )
)


# ------------------------------------------------------------
# 6. BILLING METRICS
# ------------------------------------------------------------

billing_metrics = (
    spark.table("telecom.gold.fact_billing")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("bill_id").alias(
            "total_bills"
        ),

        F.round(
            F.sum("net_amount"),
            2
        ).alias(
            "total_billed_amount"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col("bill_status") == "PAID",
                    F.col("net_amount")
                ).otherwise(0)
            ),
            2
        ).alias(
            "paid_billed_amount"
        ),

        F.sum(
            F.when(
                F.col("bill_status").isin(
                    "UNPAID",
                    "OVERDUE"
                ),
                1
            ).otherwise(0)
        ).alias(
            "unpaid_overdue_bills"
        )
    )
)


# ------------------------------------------------------------
# 7. PAYMENT METRICS
# ------------------------------------------------------------

payment_metrics = (
    spark.table("telecom.gold.fact_payments")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("payment_id").alias(
            "total_payments"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col("payment_status") == "SUCCESS",
                    F.col("payment_amount")
                ).otherwise(0)
            ),
            2
        ).alias(
            "total_successful_payments"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_payments"
        )
    )
)


# ------------------------------------------------------------
# 8. COMPLAINT METRICS
# ------------------------------------------------------------

complaint_metrics = (
    spark.table("telecom.gold.fact_complaints")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("complaint_id").alias(
            "total_complaints"
        ),

        F.sum(
            F.when(
                F.col("complaint_status").isin(
                    "RESOLVED",
                    "CLOSED"
                ),
                1
            ).otherwise(0)
        ).alias(
            "resolved_complaints"
        ),

        F.sum(
            F.when(
                F.col("priority").isin(
                    "HIGH",
                    "CRITICAL"
                ),
                1
            ).otherwise(0)
        ).alias(
            "high_priority_complaints"
        )
    )
)


# ------------------------------------------------------------
# 9. COMBINE ALL METRICS
# ------------------------------------------------------------

customer_360 = (
    customer_base

    .join(
        subscription_metrics,
        "customer_id",
        "left"
    )

    .join(
        call_metrics,
        "customer_id",
        "left"
    )

    .join(
        sms_metrics,
        "customer_id",
        "left"
    )

    .join(
        data_metrics,
        "customer_id",
        "left"
    )

    .join(
        billing_metrics,
        "customer_id",
        "left"
    )

    .join(
        payment_metrics,
        "customer_id",
        "left"
    )

    .join(
        complaint_metrics,
        "customer_id",
        "left"
    )
)


# ------------------------------------------------------------
# 10. HANDLE NULL METRICS
# ------------------------------------------------------------

metric_columns = [
    "total_subscriptions",
    "active_subscriptions",
    "cancelled_subscriptions",
    "total_calls",
    "total_call_minutes",
    "total_call_charges",
    "total_sms",
    "total_sms_charges",
    "total_data_mb",
    "total_data_gb",
    "total_data_charges",
    "total_bills",
    "total_billed_amount",
    "paid_billed_amount",
    "unpaid_overdue_bills",
    "total_payments",
    "total_successful_payments",
    "failed_payments",
    "total_complaints",
    "resolved_complaints",
    "high_priority_complaints"
]

for column_name in metric_columns:
    customer_360 = customer_360.withColumn(
        column_name,
        F.coalesce(
            F.col(column_name),
            F.lit(0)
        )
    )


# ------------------------------------------------------------
# 11. DERIVED BUSINESS METRICS
# ------------------------------------------------------------

customer_360 = (
    customer_360

    .withColumn(
        "outstanding_amount",
        F.round(
            F.col("total_billed_amount") -
            F.col("total_successful_payments"),
            2
        )
    )

    .withColumn(
        "payment_success_rate",
        F.round(
            F.when(
                F.col("total_payments") > 0,
                (
                    F.col("total_successful_payments") /
                    F.col("total_payments")
                ) * 100
            ).otherwise(0),
            2
        )
    )

    .withColumn(
        "complaint_resolution_rate",
        F.round(
            F.when(
                F.col("total_complaints") > 0,
                (
                    F.col("resolved_complaints") /
                    F.col("total_complaints")
                ) * 100
            ).otherwise(0),
            2
        )
    )

    .withColumn(
        "churn_risk_score",
        (
            F.when(
                F.col("cancelled_subscriptions") > 0,
                30
            ).otherwise(0)

            + F.when(
                F.col("unpaid_overdue_bills") > 0,
                20
            ).otherwise(0)

            + F.when(
                F.col("high_priority_complaints") > 0,
                20
            ).otherwise(0)

            + F.when(
                F.col("total_complaints") >= 2,
                15
            ).otherwise(0)

            + F.when(
                F.col("failed_payments") >= 2,
                15
            ).otherwise(0)
        )
    )

    .withColumn(
        "churn_risk_level",
        F.when(
            F.col("churn_risk_score") >= 60,
            "HIGH"
        )
        .when(
            F.col("churn_risk_score") >= 30,
            "MEDIUM"
        )
        .otherwise(
            "LOW"
        )
    )
)


# ------------------------------------------------------------
# 12. WRITE CUSTOMER 360
# ------------------------------------------------------------

(
    customer_360.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.customer_360")
)

print(
    f"✅ customer_360 created: "
    f"{customer_360.count():,} customers"
)

display(customer_360.limit(10))

In [0]:
# ============================================================
# CUSTOMER 360 — VALIDATION
# ============================================================

customer_360_check = spark.table(
    "telecom.gold.customer_360"
)

total_customers = customer_360_check.count()

duplicate_customers = (
    customer_360_check
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

null_customer_ids = (
    customer_360_check
    .filter(F.col("customer_id").isNull())
    .count()
)

negative_outstanding = (
    customer_360_check
    .filter(F.col("outstanding_amount") < 0)
    .count()
)

invalid_risk_levels = (
    customer_360_check
    .filter(
        ~F.col("churn_risk_level").isin(
            "LOW",
            "MEDIUM",
            "HIGH"
        )
    )
    .count()
)

print("=" * 70)
print("CUSTOMER 360 VALIDATION")
print("=" * 70)

print(f"Total customers       : {total_customers:,}")
print(f"Duplicate customers   : {duplicate_customers}")
print(f"Null customer IDs     : {null_customer_ids}")
print(f"Negative outstanding  : {negative_outstanding}")
print(f"Invalid risk levels   : {invalid_risk_levels}")

if (
    total_customers == 1000
    and duplicate_customers == 0
    and null_customer_ids == 0
    and negative_outstanding == 0
    and invalid_risk_levels == 0
):
    print("=" * 70)
    print("✅ CUSTOMER 360 VALIDATION PASSED")
    print("=" * 70)
else:
    print("=" * 70)
    print("⚠️ CUSTOMER 360 NEEDS REVIEW")
    print("=" * 70)

In [0]:
# ============================================================
# GOLD — EXECUTIVE KPI VIEW
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_executive_kpis AS

SELECT
    COUNT(DISTINCT customer_id) AS total_customers,

    SUM(active_subscriptions) AS active_subscriptions,

    ROUND(SUM(total_billed_amount), 2)
        AS total_billed_revenue,

    ROUND(SUM(total_successful_payments), 2)
        AS total_collected_revenue,

    ROUND(SUM(outstanding_amount), 2)
        AS total_outstanding_amount,

    SUM(total_calls) AS total_calls,

    ROUND(SUM(total_call_minutes), 2)
        AS total_call_minutes,

    SUM(total_sms) AS total_sms,

    ROUND(SUM(total_data_gb), 2)
        AS total_data_gb,

    SUM(total_complaints) AS total_complaints,

    SUM(high_priority_complaints)
        AS high_priority_complaints,

    SUM(
        CASE
            WHEN churn_risk_level = 'HIGH'
            THEN 1
            ELSE 0
        END
    ) AS high_risk_customers,

    SUM(
        CASE
            WHEN churn_risk_level = 'MEDIUM'
            THEN 1
            ELSE 0
        END
    ) AS medium_risk_customers,

    SUM(
        CASE
            WHEN churn_risk_level = 'LOW'
            THEN 1
            ELSE 0
        END
    ) AS low_risk_customers

FROM telecom.gold.customer_360
""")

print("✅ vw_executive_kpis created")

display(
    spark.table("telecom.gold.vw_executive_kpis")
)

In [0]:
# ============================================================
# GOLD — MONTHLY REVENUE ANALYSIS
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_revenue_analysis AS

SELECT
    d.year,
    d.month,
    d.month_name,

    CONCAT(
        CAST(d.year AS STRING),
        '-',
        LPAD(CAST(d.month AS STRING), 2, '0')
    ) AS year_month,

    COUNT(DISTINCT b.bill_id) AS total_bills,

    ROUND(
        SUM(b.total_amount),
        2
    ) AS gross_billed_amount,

    ROUND(
        SUM(b.tax_amount),
        2
    ) AS total_tax,

    ROUND(
        SUM(b.discount_amount),
        2
    ) AS total_discounts,

    ROUND(
        SUM(b.net_amount),
        2
    ) AS net_billed_amount,

    COUNT(
        CASE
            WHEN b.bill_status = 'PAID'
            THEN 1
        END
    ) AS paid_bills,

    COUNT(
        CASE
            WHEN b.bill_status = 'UNPAID'
            THEN 1
        END
    ) AS unpaid_bills,

    COUNT(
        CASE
            WHEN b.bill_status = 'OVERDUE'
            THEN 1
        END
    ) AS overdue_bills

FROM telecom.gold.fact_billing b

LEFT JOIN telecom.gold.dim_date d
    ON b.date_key = d.date_key

GROUP BY
    d.year,
    d.month,
    d.month_name

ORDER BY
    d.year,
    d.month
""")

print("✅ vw_revenue_analysis created")

display(
    spark.table("telecom.gold.vw_revenue_analysis")
)

In [0]:
# ============================================================
# GOLD — USAGE ANALYSIS
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_usage_analysis AS

WITH call_usage AS (

    SELECT
        d.year,
        d.month,
        d.month_name,

        CONCAT(
            CAST(d.year AS STRING),
            '-',
            LPAD(CAST(d.month AS STRING), 2, '0')
        ) AS year_month,

        COUNT(c.call_id) AS total_calls,

        ROUND(
            SUM(c.call_duration_seconds) / 60,
            2
        ) AS total_call_minutes,

        ROUND(
            SUM(c.call_charges),
            2
        ) AS call_charges

    FROM telecom.gold.fact_call_usage c

    LEFT JOIN telecom.gold.dim_date d
        ON c.date_key = d.date_key

    GROUP BY
        d.year,
        d.month,
        d.month_name

),

sms_usage AS (

    SELECT
        d.year,
        d.month,

        SUM(s.sms_count) AS total_sms,

        ROUND(
            SUM(s.sms_charges),
            2
        ) AS sms_charges

    FROM telecom.gold.fact_sms_usage s

    LEFT JOIN telecom.gold.dim_date d
        ON s.date_key = d.date_key

    GROUP BY
        d.year,
        d.month

),

data_usage AS (

    SELECT
        d.year,
        d.month,

        ROUND(
            SUM(u.data_consumed_mb) / 1024,
            2
        ) AS total_data_gb,

        ROUND(
            SUM(u.data_charges),
            2
        ) AS data_charges

    FROM telecom.gold.fact_data_usage u

    LEFT JOIN telecom.gold.dim_date d
        ON u.date_key = d.date_key

    GROUP BY
        d.year,
        d.month

)

SELECT

    c.year,
    c.month,
    c.month_name,
    c.year_month,

    c.total_calls,
    c.total_call_minutes,
    c.call_charges,

    COALESCE(s.total_sms, 0)
        AS total_sms,

    COALESCE(s.sms_charges, 0)
        AS sms_charges,

    COALESCE(u.total_data_gb, 0)
        AS total_data_gb,

    COALESCE(u.data_charges, 0)
        AS data_charges,

    ROUND(
        c.call_charges
        + COALESCE(s.sms_charges, 0)
        + COALESCE(u.data_charges, 0),
        2
    ) AS total_usage_charges

FROM call_usage c

LEFT JOIN sms_usage s
    ON c.year = s.year
    AND c.month = s.month

LEFT JOIN data_usage u
    ON c.year = u.year
    AND c.month = u.month

ORDER BY
    c.year,
    c.month
""")

print("✅ vw_usage_analysis created")

display(
    spark.table("telecom.gold.vw_usage_analysis")
)

In [0]:
# ============================================================
# GOLD — PLAN PERFORMANCE
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_plan_performance AS

WITH subscription_metrics AS (

    SELECT
        plan_id,

        COUNT(DISTINCT subscription_id)
            AS total_subscriptions,

        COUNT(
            DISTINCT CASE
                WHEN subscription_status = 'ACTIVE'
                THEN subscription_id
            END
        ) AS active_subscriptions,

        COUNT(
            DISTINCT CASE
                WHEN subscription_status = 'CANCELLED'
                THEN subscription_id
            END
        ) AS cancelled_subscriptions

    FROM telecom.gold.dim_subscription

    GROUP BY plan_id
),

billing_metrics AS (

    SELECT
        plan_id,

        COUNT(DISTINCT bill_id)
            AS total_bills,

        ROUND(
            SUM(net_amount),
            2
        ) AS total_revenue

    FROM telecom.gold.fact_billing

    GROUP BY plan_id
),

usage_metrics AS (

    SELECT
        s.plan_id,

        COUNT(DISTINCT c.call_id)
            AS total_calls,

        ROUND(
            SUM(c.call_duration_seconds) / 60,
            2
        ) AS total_call_minutes

    FROM telecom.gold.fact_call_usage c

    INNER JOIN telecom.gold.dim_subscription s
        ON c.subscription_id = s.subscription_id

    GROUP BY s.plan_id
),

data_metrics AS (

    SELECT
        s.plan_id,

        ROUND(
            SUM(u.data_consumed_mb) / 1024,
            2
        ) AS total_data_gb

    FROM telecom.gold.fact_data_usage u

    INNER JOIN telecom.gold.dim_subscription s
        ON u.subscription_id = s.subscription_id

    GROUP BY s.plan_id
)

SELECT

    p.plan_id,
    p.plan_name,
    p.plan_type,
    p.monthly_charge,

    COALESCE(sm.total_subscriptions, 0)
        AS total_subscriptions,

    COALESCE(sm.active_subscriptions, 0)
        AS active_subscriptions,

    COALESCE(sm.cancelled_subscriptions, 0)
        AS cancelled_subscriptions,

    COALESCE(bm.total_bills, 0)
        AS total_bills,

    COALESCE(bm.total_revenue, 0)
        AS total_revenue,

    COALESCE(um.total_calls, 0)
        AS total_calls,

    COALESCE(um.total_call_minutes, 0)
        AS total_call_minutes,

    COALESCE(dm.total_data_gb, 0)
        AS total_data_gb

FROM telecom.gold.dim_plan p

LEFT JOIN subscription_metrics sm
    ON p.plan_id = sm.plan_id

LEFT JOIN billing_metrics bm
    ON p.plan_id = bm.plan_id

LEFT JOIN usage_metrics um
    ON p.plan_id = um.plan_id

LEFT JOIN data_metrics dm
    ON p.plan_id = dm.plan_id

ORDER BY
    total_revenue DESC
""")

print("✅ vw_plan_performance created")

display(
    spark.table("telecom.gold.vw_plan_performance")
)

In [0]:
# ============================================================
# GOLD — COMPLAINT ANALYSIS
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_complaint_analysis AS

SELECT
    c.category_id,
    COALESCE(cat.category_name, 'Unknown') AS category_name,

    COUNT(DISTINCT c.complaint_id)
        AS total_complaints,

    SUM(
        CASE
            WHEN c.complaint_status = 'OPEN'
            THEN 1 ELSE 0
        END
    ) AS open_complaints,

    SUM(
        CASE
            WHEN c.complaint_status = 'IN_PROGRESS'
            THEN 1 ELSE 0
        END
    ) AS in_progress_complaints,

    SUM(
        CASE
            WHEN c.complaint_status IN ('RESOLVED', 'CLOSED')
            THEN 1 ELSE 0
        END
    ) AS resolved_complaints,

    SUM(
        CASE
            WHEN c.priority = 'HIGH'
            THEN 1 ELSE 0
        END
    ) AS high_priority_complaints,

    SUM(
        CASE
            WHEN c.priority = 'CRITICAL'
            THEN 1 ELSE 0
        END
    ) AS critical_complaints,

    ROUND(
        AVG(
            CASE
                WHEN c.resolution_date IS NOT NULL
                THEN DATEDIFF(
                    TO_DATE(c.resolution_date),
                    TO_DATE(c.complaint_date)
                )
            END
        ),
        2
    ) AS avg_resolution_days

FROM telecom.gold.fact_complaints c

LEFT JOIN telecom.silver.complaint_categories cat
    ON c.category_id = cat.category_id

GROUP BY
    c.category_id,
    cat.category_name

ORDER BY
    total_complaints DESC
""")

print("✅ vw_complaint_analysis created")

display(
    spark.table("telecom.gold.vw_complaint_analysis")
)

In [0]:
# ============================================================
# GOLD — CHURN RISK ANALYSIS
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_churn_risk AS

SELECT
    customer_id,
    customer_name,
    age,
    gender,
    customer_status,

    current_plan,
    current_plan_type,

    total_subscriptions,
    active_subscriptions,
    cancelled_subscriptions,

    total_calls,
    total_call_minutes,

    total_sms,

    total_data_gb,

    total_billed_amount,
    total_successful_payments,
    outstanding_amount,

    total_payments,
    failed_payments,
    payment_success_rate,

    total_complaints,
    resolved_complaints,
    high_priority_complaints,
    complaint_resolution_rate,

    churn_risk_score,
    churn_risk_level,

    customer_tenure_days,
    customer_tenure_years

FROM telecom.gold.customer_360

ORDER BY
    churn_risk_score DESC,
    outstanding_amount DESC
""")

print("✅ vw_churn_risk created")

display(
    spark.table("telecom.gold.vw_churn_risk")
    .limit(20)
)

In [0]:
# ============================================================
# CUSTOMER 360 — FIX PAYMENT SUCCESS RATE
# ============================================================

payment_metrics_corrected = (
    spark.table("telecom.gold.fact_payments")
    .groupBy("customer_id")
    .agg(
        F.countDistinct("payment_id").alias(
            "total_payments"
        ),

        F.countDistinct(
            F.when(
                F.col("payment_status") == "SUCCESS",
                F.col("payment_id")
            )
        ).alias(
            "successful_payment_count"
        ),

        F.round(
            F.sum(
                F.when(
                    F.col("payment_status") == "SUCCESS",
                    F.col("payment_amount")
                ).otherwise(0)
            ),
            2
        ).alias(
            "total_successful_payments"
        ),

        F.sum(
            F.when(
                F.col("payment_status") == "FAILED",
                1
            ).otherwise(0)
        ).alias(
            "failed_payments"
        )
    )
)

customer_360_fixed = (
    spark.table("telecom.gold.customer_360")
    .drop(
        "total_payments",
        "successful_payment_count",
        "total_successful_payments",
        "failed_payments",
        "payment_success_rate"
    )
    .join(
        payment_metrics_corrected,
        "customer_id",
        "left"
    )
    .fillna({
        "total_payments": 0,
        "successful_payment_count": 0,
        "total_successful_payments": 0,
        "failed_payments": 0
    })
    .withColumn(
        "payment_success_rate",
        F.round(
            F.when(
                F.col("total_payments") > 0,
                (
                    F.col("successful_payment_count") /
                    F.col("total_payments")
                ) * 100
            ).otherwise(0),
            2
        )
    )
)

(
    customer_360_fixed.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("telecom.gold.customer_360")
)

print(
    f"✅ customer_360 corrected: "
    f"{customer_360_fixed.count():,} customers"
)

display(
    customer_360_fixed
    .select(
        "customer_id",
        "total_payments",
        "successful_payment_count",
        "failed_payments",
        "payment_success_rate"
    )
    .limit(20)
)

In [0]:
# ============================================================
# REFRESH CHURN RISK VIEW
# ============================================================

spark.sql("""
CREATE OR REPLACE VIEW telecom.gold.vw_churn_risk AS

SELECT
    customer_id,
    customer_name,
    age,
    gender,
    customer_status,

    current_plan,
    current_plan_type,

    total_subscriptions,
    active_subscriptions,
    cancelled_subscriptions,

    total_calls,
    total_call_minutes,
    total_sms,
    total_data_gb,

    total_billed_amount,
    total_successful_payments,
    outstanding_amount,

    total_payments,
    successful_payment_count,
    failed_payments,
    payment_success_rate,

    total_complaints,
    resolved_complaints,
    high_priority_complaints,
    complaint_resolution_rate,

    churn_risk_score,
    churn_risk_level,

    customer_tenure_days,
    customer_tenure_years

FROM telecom.gold.customer_360

ORDER BY
    churn_risk_score DESC,
    outstanding_amount DESC
""")

print("✅ vw_churn_risk refreshed")

display(
    spark.table("telecom.gold.vw_churn_risk")
    .limit(20)
)

In [0]:
# ============================================================
# FINAL GOLD LAYER VALIDATION
# ============================================================

expected_gold = {
    "dim_date": 1095,
    "dim_customer": 1000,
    "dim_plan": 5,
    "dim_service": 10,
    "dim_subscription": 1242,
    "dim_service_area": 50,

    "fact_call_usage": 50000,
    "fact_sms_usage": 100000,
    "fact_data_usage": 75000,
    "fact_billing": 5039,
    "fact_payments": 4074,
    "fact_complaints": 500,

    "customer_360": 1000
}

validation_results = []

for table_name, expected_count in expected_gold.items():

    full_name = f"telecom.gold.{table_name}"

    try:
        actual_count = spark.table(full_name).count()

        validation_results.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": actual_count,
            "status": (
                "PASS"
                if actual_count == expected_count
                else "FAIL"
            )
        })

    except Exception as e:

        validation_results.append({
            "table_name": table_name,
            "expected_records": expected_count,
            "actual_records": 0,
            "status": "MISSING"
        })

gold_final_validation = spark.createDataFrame(
    validation_results
)

display(
    gold_final_validation.orderBy("table_name")
)

total_tables = len(validation_results)

passed_tables = sum(
    1
    for x in validation_results
    if x["status"] == "PASS"
)

print("=" * 70)
print("FINAL GOLD LAYER VALIDATION")
print("=" * 70)
print(f"Expected tables : {total_tables}")
print(f"Passed tables   : {passed_tables}")

if passed_tables == total_tables:
    print("=" * 70)
    print("✅ ALL GOLD TABLES VALIDATED SUCCESSFULLY")
    print("=" * 70)
else:
    print("=" * 70)
    print("⚠️ GOLD VALIDATION NEEDS REVIEW")
    print("=" * 70)